In [1]:
import time
import requests
from collections import deque
from river import compose, preprocessing, linear_model, metrics

In [2]:
API_KEY = "d6l9gmpr01qr0gn6cle0d6l9gmpr01qr0gn6cleg"
SYMBOL = "AAPL"

In [3]:
quote_url = "https://finnhub.io/api/v1/quote"

In [4]:
prices = deque(maxlen=30)
volumes = deque(maxlen=30)

In [5]:
model = (
    preprocessing.StandardScaler() |
    linear_model.LogisticRegression()
)


In [6]:
metric = metrics.Accuracy()

last_prediction = None
last_features = None


In [7]:
def get_quote(symbol: str) -> dict:
    resp = requests.get(
        quote_url,
        params={"symbol": symbol, "token": API_KEY},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()

In [8]:
def build_features(current_price: float) -> dict | None:
    if len(prices) < 6:
        return None

    ret_1 = (prices[-1] - prices[-2]) / prices[-2] if prices[-2] else 0.0
    ret_5 = (prices[-1] - prices[-6]) / prices[-6] if prices[-6] else 0.0

    ma_5 = sum(list(prices)[-5:]) / 5
    vol_5 = 0.0
    recent = list(prices)[-5:]
    mean_recent = sum(recent) / len(recent)
    vol_5 = (sum((p - mean_recent) ** 2 for p in recent) / len(recent)) ** 0.5

    x = {
        "ret_1": ret_1,
        "ret_5": ret_5,
        "dist_ma_5": (current_price - ma_5) / ma_5 if ma_5 else 0.0,
        "vol_5": vol_5,
    }
    return x
